## Environment Setup & Imports
Configure project paths and import core libraries alongside the modular `src/` engines.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_raw_data
from src.cleaning import clean_data, print_cleaning_report
from src.eda import generate_eda_report, print_eda_report
from src.feature_engineering import engineer_features
from src.preprocessing import split_data, get_feature_types, build_preprocessor

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)

## Data Ingestion & Hygiene Check
Load raw data and run sanitization: resolve whitespace strings in `TotalCharges`, encode the target, and drop non-predictive identifiers.

In [2]:
# 1. Ingest raw dataset (downloads automatically if not cached locally)
df_raw = load_raw_data(data_dir=PROJECT_ROOT / "data" / "raw")

# 2. Sanitize and view hygiene report
df_clean, clean_report = clean_data(df_raw)
print_cleaning_report(clean_report)

[DataLoader] Using cached raw dataset from: D:\AI-ML\NOTES\logistic-regression\telco-churn-guardian\data\raw\Telco-Customer-Churn.csv
[DataLoader] Ingested raw shape: 7043 rows, 21 columns.
               DATA SANITIZATION REPORT                     
  • Rows Retained                 : 7043
  • Columns Retained              : 20
  • TotalCharges Whitespace Fixed : 11
  • TotalCharges Imputed (with 0) : 11
  • Identifier Dropped            : customerID
  • Target Variable Encoded       : {'Yes': 1, 'No': 0}


## Automated EDA Profile
Audit target class distribution, confirm zero remaining missing values, and check baseline linear correlations with churn.

In [5]:
df_clean.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [4]:
eda_report = generate_eda_report(df_clean, target_col="Churn")
print_eda_report(eda_report)

                 AUTOMATED DATA PROFILE REPORT                   
Dataset Dimensions: 7043 rows × 19 input features
-----------------------------------------------------------------
TARGET CLASS DISTRIBUTION ('Churn'):
  • Retained (0) : 5,174 (73.46%)
  • Churned  (1) : 1,869 (26.54%)
  • Imbalance Ratio: 1 positive : 2.77 negatives
-----------------------------------------------------------------
DATA INTEGRITY & CARDINALITY:
  • Missing Values Detected : Zero missing
  • Numerical Features (4) : ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
  • Categorical Features (15): ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
-----------------------------------------------------------------
LINEAR CORRELATION WITH CHURN (Numeric Features):
  • MonthlyCharges        : +0.1934 (▲ Risk fac